# Marsh — FP&A Regional Analysis
**Deliverable 1: Comprehensive Transaction & Revenue Analysis — FY 2024**

Covers:
- Total transactions per client, per currency (count + local amount + USD equivalent)
- Client revenue totals converted to USD
- Monthly revenue trend
- Market section analysis (first character of client code)
- Geographic revenue distribution
- Full formatted Excel export → `output/Deliverable_1_FPA_Analysis.xlsx`

In [14]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import rcParams
from IPython.display import display
sys.path.insert(0, 'src')
from engine import load_all_transactions, load_fx_rates, currency_client_transactions, client_totals_usd
warnings.filterwarnings('ignore')

DATA_ROOT  = 'data'
FX_PATH    = os.path.join('data', '2023 Budget FX Rates.xlsx')
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MARSH_NAVY  = '#1C3F6E'
MARSH_BLUE  = '#2E75B6'
MARSH_LIGHT = '#BDD7EE'

rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F5F8FA',
    'axes.edgecolor':    '#CCCCCC',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'figure.dpi':        120,
})
pd.options.display.float_format = '{:,.2f}'.format

In [15]:
print('Loading transaction data...')
df_raw   = load_all_transactions(DATA_ROOT)
fx_rates = load_fx_rates(FX_PATH)

df = df_raw.copy()
df['Transaction']     = pd.to_numeric(df['Transaction'], errors='coerce')
df['Exchange_Rate']   = df['Currency'].map(fx_rates)
df['Transaction_USD'] = df['Transaction'] * df['Exchange_Rate']
df['Market_Section']  = df['Client'].str[0].str.upper()
df['Month']           = df['log_date'].dt.to_period('M')
df['Quarter']         = df['log_date'].dt.to_period('Q')

total_rev   = df['Transaction_USD'].sum()
total_txn   = len(df)
n_clients   = df['Client'].nunique()
n_countries = df['Country'].nunique()

print(f'  Rows loaded      : {total_txn:>10,}')
print(f'  Unique clients   : {n_clients:>10,}')
print(f'  Unique countries : {n_countries:>10,}')
print(f'  Date range       : {df["log_date"].min().date()} -> {df["log_date"].max().date()}')
print(f'  Total Revenue    : ${total_rev:>16,.2f} USD')

Loading transaction data...
  Rows loaded      :    314,754
  Unique clients   :     26,472
  Unique countries :        136
  Date range       : 2024-01-04 -> 2024-12-29
  Total Revenue    : $2,122,264,658.57 USD


## 1. Data Overview

In [16]:
print('=== Transaction Value Statistics (USD) ===')
display(df['Transaction_USD'].describe().rename('Transaction_USD').to_frame())

print('\n=== Revenue by Quarter ===')
qtly = (
    df.groupby('Quarter')
    .agg(Total_USD=('Transaction_USD', 'sum'), Transactions=('Transaction', 'count'))
    .reset_index()
)
qtly['Quarter'] = qtly['Quarter'].astype(str)
display(qtly)

=== Transaction Value Statistics (USD) ===


,Transaction_USD
count,"314,754.00"
mean,"6,742.61"
std,"11,146.52"
min,"-1,597.16"
25%,882.68
50%,"2,957.17"
75%,"7,971.04"
max,"329,947.37"



=== Revenue by Quarter ===


,Quarter,Total_USD,Transactions
0,2024Q1,"381,805,371.46",56859
1,2024Q2,"590,579,583.55",63007
2,2024Q3,"487,630,527.88",92682
3,2024Q4,"662,249,175.68",102206


## 2. Total Transactions per Client & Currency

In [17]:
cc = (
    df.groupby(['Client', 'Currency'])
    .agg(
        Transaction_Count=('Transaction', 'count'),
        Total_Local=('Transaction', 'sum'),
        Total_USD=('Transaction_USD', 'sum'),
    )
    .reset_index()
    .sort_values('Total_USD', ascending=False)
    .reset_index(drop=True)
)

print(f'Client-Currency combinations: {len(cc):,}')
print('\nTop 20 Client-Currency pairs by USD Revenue:')
display(
    cc.head(20).style
    .format({'Total_Local': '{:,.2f}', 'Total_USD': '${:,.2f}'})
    .background_gradient(subset='Total_USD', cmap='Blues')
    .set_caption('Top 20 Client-Currency Pairs by USD Revenue')
)

Client-Currency combinations: 297,893

Top 20 Client-Currency pairs by USD Revenue:


,Client,Currency,Transaction_Count,Total_Local,Total_USD
0,BDDDH,SZL,2,"7,590,218.08","$437,832.39"
1,HKKX3,ARS,3,"73,901,851.72","$335,917.17"
2,ACF22,MZN,2,"21,353,660.46","$335,221.72"
3,CFHKV,MMK,1,"692,890,165.25","$329,947.37"
4,LVXa1,ISK,2,"45,791,681.47","$318,395.97"
5,LNaa1,SZL,2,"4,982,612.36","$287,415.87"
6,CFHKV,VND,3,"7,071,338,848.34","$284,762.82"
7,AJKKb,NZD,2,"445,544.13","$271,826.47"
8,BDDDH,PES,1,"52,854,765.25","$270,011.74"
9,CDJKa,NGN,1,"119,469,937.75","$268,870.68"


## 3. Client Revenue Totals (USD)

In [ ]:
client_usd = (
    client_totals_usd(df, fx_rates)
    .sort_values('Total_USD', ascending=False)
    .reset_index(drop=True)
)
client_usd['Market_Section']    = client_usd['Client'].str[0].str.upper()
client_usd['Transaction_Count'] = df.groupby('Client').size().reindex(client_usd['Client']).values

print('Top 20 Clients by Revenue (USD):')
display(
    client_usd.head(20).style
    .format({'Total_USD': '${:,.2f}', 'Transaction_Count': '{:,}'})
    .background_gradient(subset='Total_USD', cmap='Blues')
    .set_caption('Client Revenue — Top 20')
)

fig, ax = plt.subplots(figsize=(12, 6))
top20 = client_usd.head(20)
ax.barh(top20['Client'][::-1], top20['Total_USD'][::-1] / 1e6,
        color=MARSH_BLUE, edgecolor='white', linewidth=0.5)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.2f}M'))
ax.set_xlabel('Revenue (USD Millions)', labelpad=10)
ax.set_title('Top 20 Clients by Revenue — FY 2024', pad=15, color=MARSH_NAVY)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

## 4. Monthly Revenue Trend

In [ ]:
monthly = (
    df.groupby('Month')
    .agg(Transaction_Count=('Transaction', 'count'), Total_USD=('Transaction_USD', 'sum'))
    .reset_index()
)
monthly['Month_Label'] = monthly['Month'].dt.strftime('%b %Y')

display(
    monthly[['Month_Label', 'Transaction_Count', 'Total_USD']]
    .rename(columns={'Month_Label': 'Month', 'Transaction_Count': 'Transactions', 'Total_USD': 'Revenue (USD)'})
    .style.format({'Revenue (USD)': '${:,.2f}', 'Transactions': '{:,}'})
    .set_caption('Monthly Revenue Summary — FY 2024')
)

fig, ax1 = plt.subplots(figsize=(13, 5))
x = range(len(monthly))
ax1.bar(x, monthly['Total_USD'] / 1e6, color=MARSH_BLUE, alpha=0.85, label='Revenue (USD M)', zorder=2)
ax1.set_xticks(x)
ax1.set_xticklabels(monthly['Month_Label'], rotation=30, ha='right', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}M'))
ax1.set_ylabel('Revenue (USD Millions)', labelpad=10)
ax1.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)

ax2 = ax1.twinx()
ax2.plot(x, monthly['Transaction_Count'], color=MARSH_NAVY, marker='o',
         linewidth=2, markersize=5, label='Transactions', zorder=3)
ax2.set_ylabel('Transaction Count', labelpad=10, color=MARSH_NAVY)
ax2.tick_params(axis='y', colors=MARSH_NAVY)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper left', fontsize=9)
ax1.set_title('Monthly Revenue & Transaction Volume — FY 2024', pad=15, color=MARSH_NAVY)
plt.tight_layout()
plt.show()

## 5. Market Section Analysis

In [ ]:
section = (
    df.groupby('Market_Section')
    .agg(
        Client_Count=('Client', 'nunique'),
        Transaction_Count=('Transaction', 'count'),
        Total_USD=('Transaction_USD', 'sum'),
    )
    .reset_index()
    .sort_values('Total_USD', ascending=False)
    .reset_index(drop=True)
)
section['Avg_per_Client_USD'] = section['Total_USD'] / section['Client_Count']

display(
    section.style
    .format({'Total_USD': '${:,.2f}', 'Avg_per_Client_USD': '${:,.2f}',
             'Transaction_Count': '{:,}', 'Client_Count': '{:,}'})
    .background_gradient(subset='Total_USD', cmap='Blues')
    .set_caption('Revenue by Market Section')
)

order   = section['Market_Section'].tolist()
sec_idx = section.set_index('Market_Section')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(order, [sec_idx.loc[s, 'Total_USD'] / 1e6 for s in order],
            color=MARSH_BLUE, edgecolor='white')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}M'))
axes[0].set_title('Total Revenue by Market Section')
axes[0].set_xlabel('Market Section')
axes[0].set_ylabel('Revenue (USD M)')

axes[1].bar(order, [sec_idx.loc[s, 'Avg_per_Client_USD'] for s in order],
            color=MARSH_NAVY, edgecolor='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[1].set_title('Average Revenue per Client by Section')
axes[1].set_xlabel('Market Section')
axes[1].set_ylabel('Avg Revenue / Client (USD)')

plt.suptitle('Market Section Analysis — FY 2024', y=1.02,
             fontsize=14, fontweight='bold', color=MARSH_NAVY)
plt.tight_layout()
plt.show()

## 6. Geographic Revenue Distribution

In [ ]:
geo = (
    df.groupby('Country')
    .agg(Transaction_Count=('Transaction', 'count'), Total_USD=('Transaction_USD', 'sum'))
    .reset_index()
    .sort_values('Total_USD', ascending=False)
    .reset_index(drop=True)
)

print(f'Countries with transactions: {len(geo)}')
display(
    geo.head(20).style
    .format({'Total_USD': '${:,.2f}', 'Transaction_Count': '{:,}'})
    .background_gradient(subset='Total_USD', cmap='Blues')
    .set_caption('Top 20 Countries by Revenue')
)

fig, ax = plt.subplots(figsize=(10, 8))
top15 = geo.head(15)
ax.barh(top15['Country'][::-1], top15['Total_USD'][::-1] / 1e6,
        color=MARSH_BLUE, edgecolor='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.1f}M'))
ax.set_xlabel('Revenue (USD Millions)', labelpad=10)
ax.set_title('Top 15 Countries by Revenue — FY 2024', pad=15, color=MARSH_NAVY)
plt.tight_layout()
plt.show()

## 7. Export — Deliverable 1 Excel Report

Generates `output/Deliverable_1_FPA_Analysis.xlsx` with 6 sheets:

| Sheet | Contents |
|---|---|
| Executive Summary | KPI cards + sheet index |
| Transactions by Client & Currency | Count, local total, USD total per pair |
| Client Revenue (USD) | Ranked totals with bar chart |
| Monthly Revenue | Table + line chart |
| Market Section Analysis | Table + bar chart |
| Geographic Analysis | Country ranking + bar chart |

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, LineChart, Reference

# ── Style constants ───────────────────────────────────────────────────────────
XL_NAVY  = '1C3F6E'
XL_BLUE  = '2E75B6'
XL_LIGHT = 'BDD7EE'
XL_ALT   = 'EBF3FB'
XL_WHITE = 'FFFFFF'
XL_DARK  = '1F2D3D'


def _fill(h):
    return PatternFill('solid', fgColor=h)


def _font(c=XL_DARK, bold=False, sz=10, nm='Calibri'):
    return Font(name=nm, size=sz, bold=bold, color=c)


def _border():
    s = Side(style='thin', color='D0D0D0')
    return Border(left=s, right=s, top=s, bottom=s)


def _write_table(ws, df, sr=1, sc=1, hdr=XL_NAVY):
    """Fully styled write — use only for small tables (< ~500 rows)."""
    b = _border()
    ws.row_dimensions[sr].height = 22
    for c, col in enumerate(df.columns, start=sc):
        cell = ws.cell(row=sr, column=c, value=col)
        cell.fill = _fill(hdr); cell.font = _font(XL_WHITE, True, 10)
        cell.border = b
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    for r, row in enumerate(df.itertuples(index=False), start=sr + 1):
        rf = _fill(XL_ALT) if (r - sr) % 2 == 0 else _fill(XL_WHITE)
        for c, val in enumerate(row, start=sc):
            cell = ws.cell(row=r, column=c, value=val)
            cell.fill = rf; cell.border = b; cell.font = _font()
            if isinstance(val, float):
                cell.number_format = '#,##0.00'
                cell.alignment = Alignment(horizontal='right', vertical='center')
            elif isinstance(val, int):
                cell.number_format = '#,##0'
                cell.alignment = Alignment(horizontal='right', vertical='center')
            else:
                cell.alignment = Alignment(horizontal='left', vertical='center')


def _write_fast(ws, df, col_widths=None):
    """Header styling + ws.append() for large tables. col_widths: list of ints."""
    b = _border()
    ws.row_dimensions[1].height = 22
    for c, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=c, value=col)
        cell.fill = _fill(XL_NAVY); cell.font = _font(XL_WHITE, True, 10)
        cell.alignment = Alignment(horizontal='center', vertical='center'); cell.border = b
    for row in df.itertuples(index=False):
        ws.append(list(row))
    if col_widths:
        for i, w in enumerate(col_widths, start=1):
            ws.column_dimensions[ws.cell(1, i).column_letter].width = w


def _aw(ws, lo=8, hi=45):
    for col in ws.columns:
        w = max((len(str(c.value or '')) for c in col), default=lo)
        ws.column_dimensions[col[0].column_letter].width = min(max(w + 2, lo), hi)


wb = openpyxl.Workbook()

# ── Sheet 1: Executive Summary ────────────────────────────────────────────────
ws_sum = wb.active
ws_sum.title = 'Executive Summary'
ws_sum.sheet_view.showGridLines = False

for rh, val, sz, bold in [
    (1, '',  8,  False),
    (2, 'MARSH  |  FP&A Regional Analysis — FY 2024', 20, True),
    (3, 'Comprehensive Transaction & Revenue Report', 12, False),
    (4, '', 8, False),
]:
    ws_sum.merge_cells(f'A{rh}:J{rh}')
    cell = ws_sum.cell(row=rh, column=1, value=val)
    cell.fill = _fill(XL_NAVY)
    cell.font = Font(name='Calibri', size=sz, bold=bold, color=XL_WHITE)
    cell.alignment = Alignment(horizontal='center', vertical='center')
    ws_sum.row_dimensions[rh].height = 8 if not val else (48 if sz > 15 else 22)

ws_sum.row_dimensions[6].height = 50
ws_sum.row_dimensions[7].height = 20
ws_sum.row_dimensions[8].height = 8

kpis = [
    (f'${total_rev / 1e6:.1f} M', 'Total Revenue (USD)',  'B', 'C'),
    (f'{total_txn:,}',             'Total Transactions',   'D', 'E'),
    (f'{n_clients:,}',             'Unique Clients',       'F', 'G'),
    (f'{n_countries:,}',           'Countries',            'H', 'I'),
]
for value, label, c1, c2 in kpis:
    ws_sum.merge_cells(f'{c1}6:{c2}6')
    ws_sum.merge_cells(f'{c1}7:{c2}7')
    v = ws_sum[f'{c1}6']
    v.value = value; v.fill = _fill(XL_BLUE)
    v.font = Font(name='Calibri', size=22, bold=True, color=XL_WHITE)
    v.alignment = Alignment(horizontal='center', vertical='center')
    l = ws_sum[f'{c1}7']
    l.value = label; l.fill = _fill(XL_LIGHT)
    l.font = Font(name='Calibri', size=10, bold=True, color=XL_NAVY)
    l.alignment = Alignment(horizontal='center', vertical='center')

ws_sum.merge_cells('A10:J10')
h = ws_sum['A10']
h.value = '  REPORT CONTENTS'; h.fill = _fill(XL_NAVY)
h.font = _font(XL_WHITE, True, 11); h.alignment = Alignment(horizontal='left', vertical='center')
ws_sum.row_dimensions[10].height = 22

idx_items = [
    ('Txns by Client & Currency',  'Count and total amount per client-currency pair'),
    ('Client Revenue (USD)',        'All clients ranked by USD revenue with embedded chart'),
    ('Monthly Revenue',             'Revenue and transaction volume by month with line chart'),
    ('Market Section Analysis',     'Revenue by market section (first character of client code)'),
    ('Geographic Analysis',         'Revenue ranked by country with bar chart'),
]
for i, (sn, desc) in enumerate(idx_items, start=11):
    ws_sum.row_dimensions[i].height = 18
    rf = _fill(XL_ALT) if i % 2 == 0 else _fill(XL_WHITE)
    for c in range(1, 11): ws_sum.cell(row=i, column=c).fill = rf
    ws_sum.cell(row=i, column=1, value=f'  {i - 10}.').font = _font(XL_DARK, sz=10)
    n = ws_sum.cell(row=i, column=2, value=sn)
    n.font = Font(name='Calibri', size=10, bold=True, color=XL_BLUE, underline='single')
    ws_sum.merge_cells(f'C{i}:J{i}')
    ws_sum.cell(row=i, column=3, value=desc).font = _font()

ws_sum.column_dimensions['A'].width = 5
ws_sum.column_dimensions['B'].width = 36
for c in ['C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']:
    ws_sum.column_dimensions[c].width = 14

# ── Sheet 2: Txns by Client & Currency  (297 K rows — fast path) ─────────────
ws_cc = wb.create_sheet('Txns by Client & Currency')
ws_cc.sheet_view.showGridLines = False
ws_cc.freeze_panes = 'A2'

cc_e = cc.copy()
cc_e.columns = ['Client', 'Currency', 'Transaction Count', 'Total Amount (Local)', 'Total Amount (USD)']
_write_fast(ws_cc, cc_e, col_widths=[14, 12, 18, 22, 22])

# ── Sheet 3: Client Revenue (USD)  (26 K rows — fast path) ───────────────────
ws_cli = wb.create_sheet('Client Revenue (USD)')
ws_cli.sheet_view.showGridLines = False
ws_cli.freeze_panes = 'A2'

cli_e = client_usd[['Client', 'Total_USD', 'Transaction_Count', 'Market_Section']].copy()
cli_e.insert(0, 'Rank', range(1, len(cli_e) + 1))
cli_e.columns = ['Rank', 'Client', 'Total Revenue (USD)', 'Transaction Count', 'Market Section']
_write_fast(ws_cli, cli_e, col_widths=[8, 14, 22, 18, 16])

ch = BarChart()
ch.type = 'bar'; ch.title = 'Top 20 Clients by Revenue (USD)'; ch.style = 10
ch.y_axis.title = 'Client'; ch.x_axis.title = 'Revenue (USD)'; ch.width = 22; ch.height = 16
tn = min(21, len(cli_e) + 1)
ch.add_data(Reference(ws_cli, min_col=3, min_row=1, max_row=tn), titles_from_data=True)
ch.set_categories(Reference(ws_cli, min_col=2, min_row=2, max_row=tn))
ws_cli.add_chart(ch, 'G2')

# ── Sheet 4: Monthly Revenue  (12 rows — full styling) ───────────────────────
ws_mon = wb.create_sheet('Monthly Revenue')
ws_mon.sheet_view.showGridLines = False

mon_e = monthly[['Month_Label', 'Transaction_Count', 'Total_USD']].copy()
mon_e.columns = ['Month', 'Transaction Count', 'Total Revenue (USD)']
_write_table(ws_mon, mon_e)
for r in range(2, len(mon_e) + 2):
    ws_mon.cell(r, 2).number_format = '#,##0'
    ws_mon.cell(r, 3).number_format = '"$"#,##0.00'
_aw(ws_mon)

ch2 = LineChart()
ch2.title = 'Monthly Revenue — FY 2024'; ch2.style = 10
ch2.y_axis.title = 'Revenue (USD)'; ch2.x_axis.title = 'Month'; ch2.width = 22; ch2.height = 14
nm = len(mon_e)
ch2.add_data(Reference(ws_mon, min_col=3, min_row=1, max_row=nm + 1), titles_from_data=True)
ch2.set_categories(Reference(ws_mon, min_col=1, min_row=2, max_row=nm + 1))
ws_mon.add_chart(ch2, 'E2')

# ── Sheet 5: Market Section Analysis  (15 rows — full styling) ───────────────
ws_sec = wb.create_sheet('Market Section Analysis')
ws_sec.sheet_view.showGridLines = False

sec_e = section.copy()
sec_e.columns = ['Market Section', 'Unique Clients', 'Transaction Count',
                 'Total Revenue (USD)', 'Avg Revenue / Client (USD)']
_write_table(ws_sec, sec_e)
for r in range(2, len(sec_e) + 2):
    ws_sec.cell(r, 2).number_format = '#,##0'
    ws_sec.cell(r, 3).number_format = '#,##0'
    ws_sec.cell(r, 4).number_format = '"$"#,##0.00'
    ws_sec.cell(r, 5).number_format = '"$"#,##0.00'
_aw(ws_sec)

ch3 = BarChart()
ch3.title = 'Total Revenue by Market Section'; ch3.style = 10
ch3.y_axis.title = 'Revenue (USD)'; ch3.x_axis.title = 'Market Section'; ch3.width = 20; ch3.height = 14
ns = len(sec_e)
ch3.add_data(Reference(ws_sec, min_col=4, min_row=1, max_row=ns + 1), titles_from_data=True)
ch3.set_categories(Reference(ws_sec, min_col=1, min_row=2, max_row=ns + 1))
ws_sec.add_chart(ch3, 'G2')

# ── Sheet 6: Geographic Analysis  (136 rows — full styling) ──────────────────
ws_geo = wb.create_sheet('Geographic Analysis')
ws_geo.sheet_view.showGridLines = False
ws_geo.freeze_panes = 'A2'

geo_e = geo.copy()
geo_e.insert(0, 'Rank', range(1, len(geo_e) + 1))
geo_e.columns = ['Rank', 'Country', 'Transaction Count', 'Total Revenue (USD)']
_write_table(ws_geo, geo_e)
for r in range(2, len(geo_e) + 2):
    ws_geo.cell(r, 3).number_format = '#,##0'
    ws_geo.cell(r, 4).number_format = '"$"#,##0.00'
_aw(ws_geo)

ch4 = BarChart()
ch4.type = 'bar'; ch4.title = 'Top 20 Countries by Revenue (USD)'; ch4.style = 10
ch4.y_axis.title = 'Country'; ch4.x_axis.title = 'Revenue (USD)'; ch4.width = 20; ch4.height = 15
tg = min(21, len(geo_e) + 1)
ch4.add_data(Reference(ws_geo, min_col=4, min_row=1, max_row=tg), titles_from_data=True)
ch4.set_categories(Reference(ws_geo, min_col=2, min_row=2, max_row=tg))
ws_geo.add_chart(ch4, 'F2')

# ── Save ──────────────────────────────────────────────────────────────────────
out = os.path.join(OUTPUT_DIR, 'Deliverable_1_FPA_Analysis.xlsx')
wb.save(out)
print(f'Report saved -> {out}')

Report saved -> output\Deliverable_1_FPA_Analysis.xlsx
